In [ ]:
from datetime import datetime
import os 

In [ ]:
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input"

results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

scratch =  "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

#!mkdir {scratch}

In [ ]:
!bash -lc 'rm -f acaf_pmerge_list.txt; for c in {1..22} X Y; do echo acaf_threshold.chr${c} >> acaf_pmerge_list.txt; done'

In [ ]:
def get_pgen_file_directories():
    
    pgen_dir = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input/acaf_pgen_files/pgen"  # directory that contains acaf_threshold.chr1.pgen etc.
    prefix = f"{pgen_dir}/acaf_threshold.chr"
    chroms = list(range(1, 23)) 

    with open("acaf_pmerge_list.txt", "w") as f:
        for c in chroms:
            f.write(f"{prefix}{c}\n")


In [ ]:
!plink2 \
  --pmerge-list acaf_pmerge_list.txt \
  --make-pgen \
  --out acaf_threshold.allchr


In [ ]:
!plink2 \
  --pfile acaf_threshold.allchr-merge \
  --maf 0.01 \
  --mac 100 \
  --geno 0.1 \
  --hwe 1e-15 \
  --mind 0.1 \
  --write-snplist \
  --write-samples \
  --no-id-header \
  --out qc_pass


In [ ]:
!plink2 \
  --pfile acaf_threshold.allchr-merge \
  --set-missing-var-ids @:# \
  --maf 0.01 \
  --mac 100 \
  --geno 0.1 \
  --hwe 1e-15 \
  --mind 0.1 \
  --write-snplist \
  --write-samples \
  --no-id-header \
  --out qc_pass


In [ ]:
#function calls

In [ ]:
%%bash

make_acaf_merge_list() {
  # Creates acaf_pmerge_list.txt with chr1–chr22
  for c in {1..22}; do
    echo "acaf_threshold.chr${c}" >> acaf_pmerge_list.txt
  done
}

# example call:
make_acaf_merge_list


In [ ]:

get_pgen_file_directories()

In [ ]:
%%bash

merge_plink() {
  local merge_list="$1"   # e.g. acaf_pmerge_list.txt
  local out_prefix="$2"   # e.g. acaf_threshold.allchr

  plink2 \
    --pmerge-list "$merge_list" \
    --make-pgen \
    --out "$out_prefix"
}

# example call:
merge_plink acaf_pmerge_list.txt acaf_threshold.allchr


In [ ]:
%%bash

qc_plink() {
  local pfile_prefix="$1"
  local out_prefix="$2"

  plink2 \
    --pfile "$pfile_prefix" \
    --maf 0.01 \
    --mac 100 \
    --geno 0.1 \
    --hwe 1e-15 \
    --mind 0.1 \
    --write-snplist \
    --write-samples \
    --no-id-header \
    --out "$out_prefix"
}

# example call:
qc_plink acaf_threshold.allchr-merge qc_pass
